# Replacement and Completeness Scores for Graph, Pruned Graph, and Subgraph 

<!-- <a target="_blank" href="https://colab.research.google.com/github/safety-research/circuit-tracer/blob/main/demos/replacement_completeness_scores.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a> -->

In this demo, you'll generate a graph and calculate the replacement and completeness scores for both the graph and subgraph.

In [ ]:
#@title Colab Setup Environment

try:
    import google.colab
    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login
    sys.path.append('repository/circuit-tracer')
    sys.path.append('repository/circuit-tracer/demos')
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import torch
from circuit_tracer.graph import compute_graph_scores, compute_subgraph_scores, create_pruned_graph
from circuit_tracer.utils.create_graph_files import prune_graph
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.hf_utils import load_transcoder_from_hub
from circuit_tracer.utils.decode_url_features import Feature

Let's first load the model and transcoders for `gemma-2-2b`.

In [ ]:
transcoder_set = "gemma"
model = "gemma-2-2b"
dtype = torch.bfloat16

transcoder, config = load_transcoder_from_hub(
    transcoder_set,
    dtype=dtype,
    lazy_encoder=False,
    lazy_decoder=True,
)

model_instance = ReplacementModel.from_pretrained_and_transcoders(
    model, transcoder, dtype=dtype
)

Next, we perform the attribution to generate the graph.

In [ ]:
prompt = "Fact: The capital of the state containing Dallas is"
max_n_logits = 10
desired_logit_prob = 0.95
max_feature_nodes = 5000
batch_size = 256

graph = attribute(
    prompt=prompt,
    model=model_instance,
    max_n_logits=max_n_logits,
    desired_logit_prob=desired_logit_prob,
    batch_size=batch_size,
    verbose=True,
    max_feature_nodes=max_feature_nodes,
)

Now, let's get the replacement and completeness scores for this whole graph.

In [ ]:
(unpruned_replacement_score, unpruned_completeness_score) = compute_graph_scores(graph)
print(f"Unpruned graph: replacement_score={unpruned_replacement_score}, completeness_score={unpruned_completeness_score}")

Next, we prune this graph. The `prune_graph` gives us the node and edge masks, but we need a Graph, so we call `create_pruned_graph`, which will give us a Graph of the same dimensions, with the pruned nodes/edges zeroed out. We keep it the same dimensions so that we can refer to the same nodes/edges in the same indexes when using it later.

In [ ]:
node_threshold = 0.7
edge_threshold = 0.9

node_mask, edge_mask, _ = (
    el.cpu() for el in prune_graph(graph, node_threshold, edge_threshold)
)
pruned_graph = create_pruned_graph(graph, node_mask, edge_mask)

Now, let's get the replacement and completeness scores for the pruned graph - how well does this pruned graph explain the model's output? It should be lower, since we pruned nodes and edges.
Remember that we zeroed out the pruned nodes/edges, so we aren't merging the weights into the error nodes.

In [ ]:
(pruned_replacement_score, pruned_completeness_score) = compute_graph_scores(pruned_graph)
print(f"Pruned graph: replacement_score={pruned_replacement_score}, completeness_score={pruned_completeness_score}")

Finally, we want to get the subgraph scores. Importantly, this is different than getting the pruned scores above, because here, we explicitly merge the non-subgraph features into error nodes.
We pass in the original graph, instead of the pruned graph node, so that we can the full context.
However, if you only have the pruned graph, you can just use that as well for an approximation, assuming that the pruned edges/nodes were not very important.

Since we've generated this graph before, we know what `node_id`s we want to pin to the subgraph. Pass this to `compute_subgraph_scores`, along with the original graph.

In [ ]:
# we know what pinned node IDs we want to include in the subgraph
subgraph_node_ids = ['27_22605_10','20_15589_10','E_26865_9','21_5943_10','23_12237_10','20_15589_9','16_25_9','14_2268_9','18_8959_10','4_13154_9','7_6861_9','19_1445_10','E_2329_7','E_6037_4','0_13727_7','6_4012_7','17_7178_10','15_4494_4','6_4662_4','4_7671_4','3_13984_4','1_1000_4','19_7477_9','18_6101_10','16_4298_10','7_691_10']

# convert these to Feature type array
pinned_features = []
for node_id in subgraph_node_ids:
    parts = node_id.split('_')
    layer = parts[0]
    # Don't include embed
    if layer == 'E':
        continue
    layer = int(layer)
    # Don't include logit nodes
    if layer > graph.cfg.n_layers:
        continue
    feature_idx = int(parts[1])
    pos = int(parts[2])
    pinned_features.append(Feature(layer=layer, pos=pos, feature_idx=feature_idx))

pinned_features = torch.tensor(pinned_features, dtype=dtype)

(subgraph_replacement_score, subgraph_completeness_score) = compute_subgraph_scores(graph, pinned_features)
print(f"Subgraph: replacement_score={subgraph_replacement_score}, completeness_score={subgraph_completeness_score}")

(pruned_subgraph_replacement_score, pruned_subgraph_completeness_score) = compute_subgraph_scores(pruned_graph, pinned_features)
print(f"Pruned subgraph: replacement_score={pruned_subgraph_replacement_score}, completeness_score={pruned_subgraph_completeness_score}")

Okay, let's finally print out all 3 scores so we can compare them easily.

In [ ]:
print(f"Unpruned graph: replacement_score={unpruned_replacement_score}, completeness_score={unpruned_completeness_score}")
print(f"Pruned graph: replacement_score={pruned_replacement_score}, completeness_score={pruned_completeness_score}")
print(f"Subgraph: replacement_score={subgraph_replacement_score}, completeness_score={subgraph_completeness_score}")
print(f"Pruned subgraph: replacement_score={pruned_subgraph_replacement_score}, completeness_score={pruned_subgraph_completeness_score}")